# Demo interactiva: evolución de ventas

Esta notebook carga el modelo final seleccionado por validación cruzada:

**SentenceTransformer `paraphrase-multilingual-MiniLM-L12-v2` + LinearSVC**.

La interfaz recibe una explicación nueva en español y devuelve una predicción entre **Peor**, **Igual** y **Mejor**.

## Antes de empezar

Ejecutá primero la celda de exportación agregada al final de la notebook de entrenamiento. Esa celda descarga el archivo `modelo_demo_ventas.zip`. En esta notebook se te pedirá subir ese ZIP.

**Importante:** los puntajes de LinearSVC son márgenes de decisión. No son probabilidades calibradas.

## 1. Instalación de dependencias

In [1]:
# SentenceTransformer genera el embedding y Gradio crea la interfaz web.
!pip -q install sentence-transformers gradio joblib

## 2. Imports y configuración

In [2]:
from pathlib import Path
import json
import shutil

import joblib
import numpy as np
import torch
import gradio as gr

from sentence_transformers import SentenceTransformer
from google.colab import files

CARPETA_MODELO = Path("/content/modelo_demo_ventas")

## 3. Subida del modelo exportado

In [3]:
# Seleccioná el archivo modelo_demo_ventas.zip generado por la notebook principal.
archivos_subidos = files.upload()

archivos_zip = [
    nombre
    for nombre in archivos_subidos
    if nombre.lower().endswith(".zip")
]

if len(archivos_zip) != 1:
    raise ValueError(
        "Debe subirse exactamente un archivo ZIP: modelo_demo_ventas.zip"
    )

archivo_zip = Path(archivos_zip[0])
CARPETA_MODELO.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(archivo_zip), str(CARPETA_MODELO))

print("Archivos disponibles:")
for ruta in sorted(CARPETA_MODELO.iterdir()):
    print("-", ruta.name)

Saving modelo_demo_ventas.zip to modelo_demo_ventas.zip
Archivos disponibles:
- clasificador_linearsvc.joblib
- metadatos_modelo.json


## 4. Carga del clasificador y los metadatos

In [4]:
ruta_clasificador = CARPETA_MODELO / "clasificador_linearsvc.joblib"
ruta_metadatos = CARPETA_MODELO / "metadatos_modelo.json"

if not ruta_clasificador.exists() or not ruta_metadatos.exists():
    raise FileNotFoundError(
        "El ZIP no contiene clasificador_linearsvc.joblib y metadatos_modelo.json"
    )

clasificador = joblib.load(ruta_clasificador)

with ruta_metadatos.open("r", encoding="utf-8") as archivo:
    metadatos = json.load(archivo)

print("Modelo:", metadatos["modelo_seleccionado"])
print("Encoder:", metadatos["encoder"])
print("Clases:", list(clasificador.classes_))
print("Mejores parámetros:", metadatos["mejores_parametros"])
print("Macro-F1 CV:", round(metadatos["macro_f1_cv"], 3))
print("Macro-F1 test:", round(metadatos["macro_f1_test"], 3))

Modelo: SentenceTransformer + LinearSVC
Encoder: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Clases: ['Igual', 'Mejor', 'Peor']
Mejores parámetros: {'C': 0.1, 'class_weight': 'balanced'}
Macro-F1 CV: 0.623
Macro-F1 test: 0.657


## 5. Carga del mismo SentenceTransformer

In [5]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

encoder = SentenceTransformer(
    metadatos["encoder"],
    device=DEVICE
)

print("Dispositivo:", DEVICE)
print("Encoder cargado correctamente.")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (4 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dispositivo: cpu
Encoder cargado correctamente.


## 6. Función de inferencia

La función reproduce el procedimiento del modelo final:

1. Recibe el texto sin aplicar una limpieza agresiva.
2. Genera un embedding normalizado de la oración completa.
3. Aplica LinearSVC.
4. Devuelve la clase y los puntajes de decisión.

Los puntajes permiten comparar qué clase quedó más cerca de la decisión del modelo, pero **no deben interpretarse como probabilidades**.

In [6]:
def predecir_evolucion(texto):
    texto = str(texto).strip()

    if not texto:
        return "Ingresá una explicación.", {}

    embedding = encoder.encode(
        [texto],
        convert_to_numpy=True,
        normalize_embeddings=metadatos["normalizar_embeddings"]
    )

    prediccion = clasificador.predict(embedding)[0]
    puntajes = clasificador.decision_function(embedding)[0]

    puntajes_por_clase = {
        str(clase): round(float(puntaje), 4)
        for clase, puntaje in zip(
            clasificador.classes_,
            puntajes
        )
    }

    return str(prediccion), puntajes_por_clase

## 7. Prueba mínima antes de abrir la interfaz

In [7]:
texto_prueba = "Hay más movimiento porque empezó la temporada de verano"
prediccion, puntajes = predecir_evolucion(texto_prueba)

print("Texto:", texto_prueba)
print("Predicción:", prediccion)
print("Puntajes de decisión:", puntajes)

Texto: Hay más movimiento porque empezó la temporada de verano
Predicción: Mejor
Puntajes de decisión: {'Igual': -0.6532, 'Mejor': 0.7028, 'Peor': -0.9205}


## 8. Interfaz web para la presentación

In [8]:
descripcion = (
    "Escribí una explicación breve sobre el motivo de la evolución de las ventas. "
    "El modelo devuelve Peor, Igual o Mejor. "
    "Los puntajes mostrados son márgenes de decisión de LinearSVC, no probabilidades."
)

demo = gr.Interface(
    fn=predecir_evolucion,
    inputs=gr.Textbox(
        lines=4,
        label="Explicación del comercio",
        placeholder=(
            "Ejemplo: Hay más clientes por el comienzo de la temporada"
        )
    ),
    outputs=[
        gr.Textbox(label="Predicción del modelo"),
        gr.JSON(label="Puntajes de decisión por clase")
    ],
    title="Clasificación de la evolución de ventas",
    description=descripcion,
    examples=[
        ["Hay más clientes por el comienzo de la temporada"],
        ["La gente está comprando menos por la falta de dinero"],
        ["Se mantiene la misma cantidad de clientes que el mes pasado"]
    ]
)

# En Google Colab, share=True crea un enlace temporal mientras la celda está activa.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d5fcc6cfeb8b79e501.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Recomendaciones para la exposición

- Ejecutá esta notebook antes de comenzar la presentación y dejá abierta la interfaz.
- Probá previamente el texto que quieran mostrar, pero durante la exposición aclaren que se trata de una inferencia sobre un ejemplo nuevo.
- La predicción no establece una causa real de las ventas: clasifica el texto según patrones aprendidos en el corpus.
- No presenten los puntajes de decisión como porcentajes de confianza.
- El enlace público de Gradio es temporal y deja de funcionar cuando se detiene la ejecución de Colab.